# Detector-oracle validation
All count numbers depend on GroundingDINO, and `car` keeps behaving oddly. Here we (1) cross-check GroundingDINO against a second open-vocab detector (**OWL-ViT**), (2) test **threshold sensitivity**, (3) break agreement down **per category**, and (4) provide a **human-annotation** scaffold for a detector-vs-human check.

**Runtime:** GPU (~20 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
else:
    !git pull
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from src.prompts import build_prompt, generate_grid
from src.pipeline import load_sdxl, generate
from src.detector import Detector
from src.scoring import count_from_detections, nms, exact_accuracy, mae
from src.config import load_config
from transformers import OwlViTProcessor, OwlViTForObjectDetection

In [ ]:
cfg = load_config('configs/exp_detval.yaml')
raw = yaml.safe_load(open('configs/exp_detval.yaml'))
gthr = raw['gdino_thresholds']; owl_thr = raw['owl_threshold']
pipe = load_sdxl(); gdino = Detector()
owl_proc = OwlViTProcessor.from_pretrained('google/owlvit-base-patch32')
owl_model = OwlViTForObjectDetection.from_pretrained('google/owlvit-base-patch32').to(pipe.device)
@torch.no_grad()
def owl_count(img, obj, thr):
    inp = owl_proc(text=[[f'a photo of a {obj}']], images=img, return_tensors='pt').to(pipe.device)
    out = owl_model(**inp)
    ts = torch.tensor([img.size[::-1]]).to(pipe.device)
    res = owl_proc.post_process_object_detection(out, threshold=thr, target_sizes=ts)[0]
    boxes = [{'label': obj, 'score': float(s), 'box': [float(x) for x in b]}
             for s, b in zip(res['scores'], res['boxes'])]
    return len(nms(boxes, 0.5))
def gdino_count(img, obj, thr):
    return count_from_detections(gdino.detect(img, [obj]), obj, thr)

In [ ]:
# Generate a validation set; score with GroundingDINO (3 thresholds) + OWL-ViT.
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
images, rows = [], []
for i, p in enumerate(grid):
    img = generate(pipe, p.text, p.seed, cfg.num_inference_steps)
    images.append(img)
    r = {'idx': i, 'obj': p.obj, 'count': p.count, 'seed': p.seed}
    for t in gthr:
        r[f'gdino_{t}'] = gdino_count(img, p.obj, t)
    r['owl'] = owl_count(img, p.obj, owl_thr)
    rows.append(r)
    if (i + 1) % 20 == 0: print(f'{i+1}/{len(grid)}')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/exp_detval.csv', index=False)
df.head()

In [ ]:
# GroundingDINO (@0.3) vs OWL-ViT agreement, overall and per category.
g = 'gdino_0.3'
def agree(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return pd.Series({'exact_match': float(np.mean(a == b)),
                      'MAE': float(np.mean(np.abs(a - b))),
                      'corr': float(np.corrcoef(a, b)[0, 1]) if a.std() and b.std() else np.nan})
print('OVERALL GroundingDINO@0.3 vs OWL-ViT:')
print(agree(df[g], df['owl']).round(3))
print('\nPER CATEGORY:')
per = df.groupby('obj').apply(lambda d: agree(d[g], d['owl'])).round(3)
print(per)
per.to_csv('results/exp_detval_percat.csv')

In [ ]:
# Threshold sensitivity: mean GroundingDINO count per category at each threshold.
sens = df.groupby('obj')[[f'gdino_{t}' for t in gthr] + ['owl']].mean().round(2)
print('mean detected count by category (GDINO thresholds + OWL):')
print(sens)
ax = sens.plot(kind='bar', figsize=(10, 4.5))
ax.set_ylabel('mean detected count'); ax.set_title('Detector counts by category & threshold')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig('results/exp_detval_thresholds.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
# HUMAN ANNOTATION scaffold. Displays n_human images with detector counts;
# fill `human` below with YOUR true counts, then run the next cell.
rng = np.random.default_rng(0)
pick = sorted(rng.choice(len(images), size=min(raw['n_human'], len(images)), replace=False))
cols = 4; nrows = (len(pick) + cols - 1) // cols
fig, axes = plt.subplots(nrows, cols, figsize=(cols * 3, nrows * 3))
axes = np.array(axes).flatten()
for ax, idx in zip(axes, pick):
    r = df.iloc[idx]
    ax.imshow(images[idx]); ax.axis('off')
    ax.set_title(f"idx {idx}: {r['count']} {r['obj']}\nGDINO {r['gdino_0.3']} | OWL {r['owl']}", fontsize=7)
for ax in axes[len(pick):]: ax.axis('off')
plt.tight_layout(); plt.savefig('results/exp_detval_human.png', dpi=80, bbox_inches='tight'); plt.show()
print('Fill in your true counts, e.g.:  human = {', pick[0], ': 2, ', pick[1], ': 3, ...}')
print('picked indices:', pick)

In [ ]:
# After filling `human = {idx: true_count, ...}` for the picked indices, run this.
human = {}   # <-- FILL THIS IN from the grid above
if human:
    idxs = list(human.keys()); tv = np.array([human[i] for i in idxs], float)
    gd = df.set_index('idx').loc[idxs, 'gdino_0.3'].to_numpy(float)
    ow = df.set_index('idx').loc[idxs, 'owl'].to_numpy(float)
    print('GroundingDINO@0.3 vs HUMAN: exact', round(float(np.mean(gd==tv)),3),
          '| MAE', round(float(np.mean(np.abs(gd-tv))),3),
          '| corr', round(float(np.corrcoef(gd,tv)[0,1]),3))
    print('OWL-ViT vs HUMAN:          exact', round(float(np.mean(ow==tv)),3),
          '| MAE', round(float(np.mean(np.abs(ow-tv))),3))
else:
    print('human dict is empty - fill it in from the grid above and re-run this cell.')

## How to read this
- **GroundingDINO@0.3 vs OWL-ViT: high per-category agreement (exact-match, low MAE)** = the count oracle is stable and our numbers are trustworthy for those categories.
- **A category with poor GDINO-OWL agreement / large threshold sensitivity (likely `car`)** = the oracle is unreliable there -> DROP that category from the quantitative claims (and note it), which also cleans up Exp #2's forest plot and Exp #3.
- **Human check:** fill in true counts for the shown sample; report GDINO-vs-human exact-match / MAE as the headline oracle validation the paper needs. If GDINO tracks humans well (except car), we cite that and proceed.